<a href="https://colab.research.google.com/github/dakshini01/ProdFusion/blob/main/codes/Data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("garments_worker_productivity.csv")

Saving garments_worker_productivity.csv to garments_worker_productivity (3).csv


In [41]:
# 2. Clean text columns first
df["department"] = df["department"].str.strip()
df["day"] = df["day"].str.strip()
df["quarter"] = df["quarter"].str.strip()

In [42]:
#Convert Date
df["date"] = pd.to_datetime(df["date"])




In [43]:
#Handle Missing WIP
df["wip_missing"] = df["wip"].isnull().astype(int)
df["wip"] = df["wip"].fillna(0)

In [44]:
#drop unnecessary columns
df = df[[
    'date',
    'actual_productivity',
    'targeted_productivity',
    'smv',
    'wip',
    'over_time',
    'incentive',
    'idle_time',
    'idle_men',
    'no_of_style_change',
    'no_of_workers',
    'team',
    'department',
    'day',
    'wip_missing'
]]

In [45]:
#Sort by date only
df = df.sort_values(["team", "date"])
df.reset_index(drop=True, inplace=True)

In [46]:
#Encode Categorical Variables
df = pd.get_dummies(df, columns=['department', 'day'], drop_first=True)


In [47]:
#Convert bool columns to int BEFORE split
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

In [55]:
#Time based Train-Test Split
# Sort by date
df = df.sort_values(by="date")

split_date = df["date"].quantile(0.8)

train = df[df["date"] < split_date]
test  = df[df["date"] >= split_date]
# Features and target
X_train = train.drop(columns=["actual_productivity", "date"])
y_train = train["actual_productivity"]

X_test = test.drop(columns=["actual_productivity", "date"])
y_test = test["actual_productivity"]

# Check shapes
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (943, 17)
Test: (254, 17)


In [56]:
#scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = [
    'targeted_productivity',
    'smv',
    'wip',
    'over_time',
    'incentive',
    'idle_time',
    'idle_men',
    'no_of_style_change',
    'no_of_workers'
]

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [57]:
#Check Final Dataset Shape
print("Full dataset shape:", df.shape)
print("Training set shape:", train.shape)
print("Test set shape:", test.shape)

Full dataset shape: (1197, 19)
Training set shape: (943, 19)
Test set shape: (254, 19)


In [58]:
#Check Final Dataset Shape
print("Remaining missing values in full dataset:")
print(df.isnull().sum().sum())

print("Remaining missing values in training:")
print(X_train.isnull().sum().sum())

print("Remaining missing values in test:")
print(X_test.isnull().sum().sum())

Remaining missing values in full dataset:
0
Remaining missing values in training:
0
Remaining missing values in test:
0


In [59]:
df.head()

,date,actual_productivity,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,team,wip_missing,department_sweing,day_Saturday,day_Sunday,day_Thursday,day_Tuesday,day_Wednesday
0,2015-01-01,0.886500,0.208151,-1.016778,-0.453931,-1.077682,-0.238643,-0.057473,-0.113005,-0.351617,-1.199268,1,1,0,0,0,1,0,0
106,2015-01-01,0.753098,0.208151,0.439527,0.030233,0.427953,-0.026297,-0.057473,-0.113005,-0.351617,0.918955,2,0,1,0,0,1,0,0
105,2015-01-01,0.755167,0.208151,-1.016778,-0.453931,-1.077682,-0.238643,-0.057473,-0.113005,-0.351617,-1.199268,2,1,0,0,0,1,0,0
910,2015-01-01,0.712205,0.208151,0.388332,-0.072148,0.571347,0.042404,-0.057473,-0.113005,-0.351617,0.873886,10,0,1,0,0,1,0,0
1,2015-01-01,0.750428,0.208151,1.190077,-0.004114,0.696816,0.042404,-0.057473,-0.113005,-0.351617,1.031626,1,0,1,0,0,1,0,0


In [60]:
#See Final Feature Columns
print("Final Feature Columns:")
print(X_train.columns)

Final Feature Columns:
Index(['targeted_productivity', 'smv', 'wip', 'over_time', 'incentive',
       'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'team',
       'wip_missing', 'department_sweing', 'day_Saturday', 'day_Sunday',
       'day_Thursday', 'day_Tuesday', 'day_Wednesday'],
      dtype='object')


In [61]:
#Confirm Time-Based Split Worked
print("Last training date:", train["date"].max())
print("First test date:", test["date"].min())

Last training date: 2015-02-25 00:00:00
First test date: 2015-02-26 00:00:00
